# 📘 Notebook 06: Pandas: Working with Data

### Course: *From Python to Deep Learning & Fine-Tuning*
**Instructor:** Ioannis Tsioulis

*Module B: Scientific Python · Notebook 2 of 3 in this module · (6 of 18 overall)*

---

## 🎯 Learning objectives

By the end of this notebook you will be able to:

- Understand the **Series** and **DataFrame**, the two core Pandas structures
- Load data into a DataFrame and inspect it
- Select, filter, and sort rows and columns
- Handle **missing values**, a constant reality in real data
- Summarise and aggregate data with **`groupby`**

> **Prerequisites:** Module A and Notebook 05 (NumPy).
>
> 🔭 **Why this matters:** before any model can learn, data must be loaded, cleaned, and explored. In practice, data scientists spend the majority of their time on exactly this. Pandas is the standard tool for it, and a clean DataFrame is the usual input to a machine-learning pipeline.

> ℹ️ **Setup note (browser kernel).** If `import pandas as pd` reports the package is missing, run once:
```python
import piplite
await piplite.install('pandas')
```

In [ ]:
import pandas as pd
import numpy as np
print("Pandas version:", pd.__version__)

## 1. The Series: a labelled column

### Intuition first
A **Series** is a one-dimensional array *with labels*. Think of it as a single column of a spreadsheet: it has values, and each value has an index (a label). It builds directly on the NumPy array you learned in Notebook 05.

In [ ]:
temperatures = pd.Series([22, 19, 25, 30],
                         index=["Mon", "Tue", "Wed", "Thu"])
print(temperatures)
print()
print("Wednesday:", temperatures["Wed"])
print("Mean:", temperatures.mean())
print("Highest:", temperatures.max())

## 2. The DataFrame: the heart of Pandas

### Intuition first
A **DataFrame** is a full table: rows and named columns, like a spreadsheet or a database table. Each column is a Series. This is the structure you will use for virtually all tabular data.

We will build a small dataset of students by hand so everything is reproducible:

In [ ]:
data = {
    "name":   ["Anna", "George", "Maria", "Kostas", "Eleni"],
    "age":    [20, 22, 21, 23, 20],
    "grade":  [8.5, 6.0, 9.2, 5.5, 7.8],
    "city":   ["Athens", "Patras", "Athens", "Thessaloniki", "Patras"],
}
df = pd.DataFrame(data)
df

### First steps with any new dataset
Before analysing data, always look at it. These methods should be the first thing you run on any DataFrame:

In [ ]:
print("First rows:")
print(df.head(3))          # the first 3 rows
print("\nShape (rows, cols):", df.shape)
print("\nColumn names:", list(df.columns))

In [ ]:
# .info() summarises columns, types, and missing values
df.info()

In [ ]:
# .describe() gives summary statistics for numeric columns
df.describe()

## 3. Selecting columns and rows

### Selecting columns
Select a single column by name (returns a Series) or several columns with a list (returns a DataFrame):

In [ ]:
print(df["name"])            # one column -> a Series
print()
print(df[["name", "grade"]]) # multiple columns -> a DataFrame

### Selecting rows with `.loc` and `.iloc`
- `.iloc[...]` selects by **integer position** (like NumPy).
- `.loc[...]` selects by **label/condition**.
Keeping these two straight is essential.

In [ ]:
print("Row at position 0 (iloc):")
print(df.iloc[0])
print("\nRows 1 to 2, columns name & grade (loc):")
print(df.loc[1:2, ["name", "grade"]])

## 4. Filtering rows by condition

This is the Pandas equivalent of the boolean masking you saw in NumPy, and one of the most-used operations in practice. Write a condition, and Pandas keeps only the rows where it is `True`.

In [ ]:
# Students who scored 8 or above
high_achievers = df[df["grade"] >= 8]
print(high_achievers)

# Combine conditions with & (and) / | (or). Each condition needs parentheses
young_top = df[(df["grade"] >= 8) & (df["age"] <= 21)]
print("\nYoung high-achievers:")
print(young_top)

In [ ]:
# Filter by a text column too
athenians = df[df["city"] == "Athens"]
print("Students from Athens:")
print(athenians)

# isin() checks membership in a set of values
two_cities = df[df["city"].isin(["Athens", "Patras"])]
print("\nFrom Athens or Patras:", len(two_cities), "students")

> ⚠️ **Common pitfalls**
>
> - Use `&` and `|` (not the words `and`/`or`) when combining Pandas conditions, and wrap **each** condition in parentheses. `df[df['a'] > 1 & df['b'] < 2]` fails; `df[(df['a'] > 1) & (df['b'] < 2)]` works.

## 5. Sorting and creating new columns

In [ ]:
# Sort by grade, highest first
print(df.sort_values("grade", ascending=False))

# Create a new column derived from existing ones
df["passed"] = df["grade"] >= 5
df["grade_out_of_100"] = df["grade"] * 10
print(df[["name", "grade", "passed", "grade_out_of_100"]])

## 6. Handling missing values

### Why this is unavoidable
Real datasets have gaps: an unanswered survey question, a failed sensor reading, a blank cell. Pandas represents missing values as `NaN` (*Not a Number*). Models cannot train on `NaN`, so dealing with it is a required preprocessing step.

Let us deliberately introduce a missing value and then handle it:

In [ ]:
df2 = df.copy()
df2.loc[1, "grade"] = np.nan   # erase one grade
print(df2[["name", "grade"]])
print("\nMissing values per column:")
print(df2.isna().sum())

In [ ]:
# Strategy 1: drop rows with missing values
print("After dropping:")
print(df2.dropna()[["name", "grade"]])

# Strategy 2: fill missing values (here, with the column mean), called imputation
mean_grade = df2["grade"].mean()
filled = df2.copy()
filled["grade"] = filled["grade"].fillna(mean_grade)
print("\nAfter filling with the mean:")
print(filled[["name", "grade"]])

> 🧠 **Dropping vs filling is a real decision, not a formality.** Dropping loses data; filling invents it. The right choice depends on how much is missing and why. We return to this judgement in the ML modules.

## 7. Aggregating with `groupby`

### Intuition first
`groupby` follows a **split, apply, combine** pattern: *split* the data into groups, *apply* a calculation to each group, and *combine* the results into a summary table. 'What is the average grade **per city**?' is a perfect `groupby` question.

In [ ]:
# Average grade and age per city
summary = df.groupby("city")[["grade", "age"]].mean()
print(summary)

# Count of students per city
print("\nStudents per city:")
print(df.groupby("city").size())

In [ ]:
# You can apply several aggregations at once with .agg()
detailed = df.groupby("city")["grade"].agg(["mean", "max", "min", "count"])
print(detailed)

---
## ✏️ Exercises

Use the `df` DataFrame defined earlier (re-run that cell first if needed).

### Exercise 1
Select and print only the `name` and `city` columns, for students older than 20.

In [ ]:
# Your solution here:


<details><summary>💡 Show solution</summary>

```python
result = df[df["age"] > 20][["name", "city"]]
print(result)
```
</details>

### Exercise 2
Add a new column `category` that is `"high"` if the grade is >= 8 and `"standard"` otherwise. (Hint: `np.where(condition, value_if_true, value_if_false)` is a clean way to do this.)

In [ ]:
import numpy as np
# Your solution here:


<details><summary>💡 Show solution</summary>

```python
df["category"] = np.where(df["grade"] >= 8, "high", "standard")
print(df[["name", "grade", "category"]])
```
</details>

### Exercise 3
Using `groupby`, compute the **highest** grade in each city. (Hint: chain `.max()` instead of `.mean()`.)

In [ ]:
# Your solution here:


<details><summary>💡 Show solution</summary>

```python
best = df.groupby("city")["grade"].max()
print(best)
```
</details>

## 🔑 Key takeaways

- A **Series** is a labelled 1-D array; a **DataFrame** is a labelled table of Series, the core data object.
- Always inspect a new dataset with `.head()`, `.info()`, `.describe()`, and `.shape`.
- Select with `[...]`, `.loc` (labels) and `.iloc` (positions); filter with boolean conditions (using `&`/`|`).
- **Missing values** (`NaN`) must be handled, by dropping or imputing, before modelling.
- `groupby` implements split, apply, combine for fast summaries; `.agg()` applies several at once.

---
**Next:** *Notebook 07: Data Visualisation*: turning numbers into plots that reveal what the data is actually saying.

---
*© Ioannis Tsioulis. *From Python to Deep Learning & Fine-Tuning*. Prepared for educational use.*